In [0]:
%run ./phase2_common

In [0]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

dbutils.widgets.dropdown("mode", "auto", ["auto", "full", "incremental"], "Mode (auto = full on first run)")
dbutils.widgets.text("api_key", "", "TMDB API key (blank = secret scope)")
dbutils.widgets.text("raw_output_dir", "/Volumes/workspace/tmdb_lakehouse/raw", "Base folder for raw batches")
dbutils.widgets.text("full_pages", "10", "Full load: discover pages (20 movies each)")
dbutils.widgets.text("lookback_days", "8", "Incremental: change window in days (1-14)")
dbutils.widgets.text("max_movies", "300", "Maximum movies to fetch")
dbutils.widgets.text("batch_id", "", "Batch ID (blank = <mode>-YYYYMMDD-HHMM)")
dbutils.widgets.dropdown("run_silver", "true", ["true", "false"], "Run Silver after Bronze?")
dbutils.widgets.dropdown("run_gold", "true", ["true", "false"], "Run Gold after Silver?")

cfg = {name: dbutils.widgets.get(name).strip() for name in (
    "mode", "api_key", "raw_output_dir", "full_pages", "lookback_days",
    "max_movies", "batch_id", "run_silver", "run_gold",
)}
full_pages = int(cfg["full_pages"] or 10)
lookback_days = int(cfg["lookback_days"] or 8)
max_movies = int(cfg["max_movies"] or 300)
if not 1 <= lookback_days <= 14:
    raise ValueError("lookback_days must be between 1 and 14")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.raw")
has_data = spark.table(BRONZE).limit(1).count() > 0
mode = cfg["mode"] if cfg["mode"] != "auto" else ("incremental" if has_data else "full")
print("Mode:", mode)


def resolve_api_key():
    if cfg["api_key"]:
        return cfg["api_key"]
    try:
        return dbutils.secrets.get(scope="tmdb-lakehouse-secrets", key="tmdb-api-key")
    except Exception:
        if os.environ.get("TMDB_API_KEY"):
            return os.environ["TMDB_API_KEY"]
    raise ValueError("No TMDB API key: set the secret tmdb-lakehouse-secrets/tmdb-api-key or the api_key widget")


api_key = resolve_api_key()
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4, backoff_factor=1, status_forcelist=(429, 500, 502, 503, 504),
)))
if api_key.count(".") == 2:
    session.headers["Authorization"] = f"Bearer {api_key}"
    auth_params = {}
else:
    auth_params = {"api_key": api_key}


def tmdb(path, **params):
    try:
        response = session.get(
            f"https://api.themoviedb.org/3{path}", params={**auth_params, **params}, timeout=30
        )
        response.raise_for_status()
        return response.json()
    except requests.RequestException as exc:
        status = getattr(exc.response, "status_code", None)
        raise RuntimeError(f"TMDB {path} failed ({status or type(exc).__name__})") from None

In [0]:
now = datetime.now(timezone.utc)
batch_id = cfg["batch_id"] or f"{mode}-{now:%Y%m%d-%H%M}"
window_start = window_end = ""

ids = []
if mode == "full":
    for page in range(1, full_pages + 1):
        ids += [m["id"] for m in tmdb("/discover/movie", sort_by="popularity.desc", page=page)["results"]]
        time.sleep(0.2)
    print(f"{len(set(ids))} distinct movies from {full_pages} discover pages")
else:
    window_start = (now.date() - timedelta(days=lookback_days)).isoformat()
    window_end = now.date().isoformat()
    page, total_found = 1, 0
    while len(ids) < max_movies:
        data = tmdb("/movie/changes", start_date=window_start, end_date=window_end, page=page)
        total_found = data.get("total_results", total_found)
        ids += [item["id"] for item in data["results"]]
        if page >= min(data.get("total_pages", 1), 500):
            break
        page += 1
        time.sleep(0.2)
    print(f"{total_found} changed movies found for {window_start} to {window_end}")

ids = list(dict.fromkeys(ids))[:max_movies]
print(f"Fetching {len(ids)} movies")

movies, skipped = [], []
for position, movie_id in enumerate(ids, 1):
    try:
        movies.append(tmdb(f"/movie/{movie_id}", append_to_response="credits,keywords,reviews"))
    except RuntimeError as exc:
        skipped.append(movie_id)
        print("skip", movie_id, exc)
    time.sleep(0.05)
    if position % 50 == 0:
        print(f"{position}/{len(ids)} fetched")
print(f"Fetched {len(movies)}; skipped {len(skipped)}")

if not movies:
    dbutils.notebook.exit(f"NO_CHANGES:{mode}")

batch_dir = f"{cfg['raw_output_dir'].rstrip('/')}/{mode}/{batch_id}"
os.makedirs(batch_dir, exist_ok=True)
output_path = f"{batch_dir}/movies_{mode}_load.json"
with open(output_path, "w", encoding="utf-8") as handle:
    json.dump({
        "load_type": mode, "window_start": window_start or None, "window_end": window_end or None,
        "record_count": len(movies), "skipped_ids": skipped, "records": movies,
    }, handle)
print("Wrote", output_path)

In [0]:
def run(notebook, **arguments):
    print("Running", notebook)
    dbutils.notebook.run(f"./{notebook}", 7200, arguments)


run(
    "phase2_raw_to_bronze", input_path=output_path, load_type=mode, batch_id=batch_id,
    source_extracted_at=now.isoformat(), window_start=window_start, window_end=window_end,
)
if cfg["run_silver"] == "true":
    for notebook in ("phase2_bronze_to_silver", "phase2_silver_child_tables", "phase2_silver_reviews"):
        run(notebook, batch_id=batch_id)
if cfg["run_gold"] == "true":
    run("phase3_gold_dimensional", batch_id=batch_id, run_optimize="true")

print("Pipeline finished for batch", batch_id)
dbutils.notebook.exit(batch_id)

In [0]:
%sql
SELECT DISTINCT batch_id, load_type, COUNT(*) AS movies
FROM workspace.tmdb_lakehouse.bronze_movie_records
GROUP BY batch_id, load_type;